In [1]:
import numpy as np

np.random.seed(42)

# Dimensions
m = 100      # samples
Dx = 200     # input features
C = 10       # classes
Da1 = 20     # hidden neurons

# Input data
X = np.random.randn(m, Dx)

In [2]:
# Random labels (0–9)
y_labels = np.random.randint(0, C, size=m)

# One-hot encoding
Y = np.zeros((m, C))
Y[np.arange(m), y_labels] = 1

In [3]:
W1 = np.random.randn(Da1, Dx) * 0.01
b1 = np.zeros((Da1, 1))

W2 = np.random.randn(C, Da1) * 0.01
b2 = np.zeros((C, 1))

In [4]:
def tanh(x):
    return np.tanh(x)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

# Transpose X to match dimensions (Dx x m)
X_T = X.T   # (200 x 100)

# Forward propagation
z1 = np.dot(W1, X_T) + b1
a1 = tanh(z1)

z2 = np.dot(W2, a1) + b2
y_hat = softmax(z2)   # (10 x 100)

In [5]:
# Convert Y to shape (10 x 100)
Y_T = Y.T

# Loss
loss = -np.sum(Y_T * np.log(y_hat + 1e-9)) / m

print("Cross-Entropy Loss:", loss)

Cross-Entropy Loss: 2.302008979827582


In [6]:
y_pred = np.argmax(y_hat, axis=0)
y_true = np.argmax(Y_T, axis=0)

In [7]:
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

Accuracy: 0.17


In [8]:
precision_list = []
recall_list = []
f1_list = []

for cls in range(C):
    TP = np.sum((y_pred == cls) & (y_true == cls))
    FP = np.sum((y_pred == cls) & (y_true != cls))
    FN = np.sum((y_pred != cls) & (y_true == cls))

    precision = TP / (TP + FP + 1e-9)
    recall = TP / (TP + FN + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)

print("Precision:", np.mean(precision_list))
print("Recall:", np.mean(recall_list))
print("F1-score:", np.mean(f1_list))

Precision: 0.1934174158899846
Recall: 0.1669119768940864
F1-score: 0.16694310603353008


In [9]:
# Backpropagation

# dz2
dz2 = y_hat - Y_T   # (10 x 100)

# dW2, db2
dW2 = (1/m) * np.dot(dz2, a1.T)
db2 = (1/m) * np.sum(dz2, axis=1, keepdims=True)

# dz1
da1 = np.dot(W2.T, dz2)
dz1 = da1 * (1 - np.power(a1, 2))

# dW1, db1
dW1 = (1/m) * np.dot(dz1, X_T.T)
db1 = (1/m) * np.sum(dz1, axis=1, keepdims=True)

In [10]:
learning_rate = 0.01

W1 -= learning_rate * dW1
b1 -= learning_rate * db1

W2 -= learning_rate * dW2
b2 -= learning_rate * db2

In [11]:
epochs = 100

for i in range(epochs):
    
    # Forward pass
    z1 = np.dot(W1, X_T) + b1
    a1 = np.tanh(z1)

    z2 = np.dot(W2, a1) + b2
    y_hat = softmax(z2)

    # Loss
    loss = -np.sum(Y_T * np.log(y_hat + 1e-9)) / m

    # Backprop
    dz2 = y_hat - Y_T
    dW2 = (1/m) * np.dot(dz2, a1.T)
    db2 = (1/m) * np.sum(dz2, axis=1, keepdims=True)

    da1 = np.dot(W2.T, dz2)
    dz1 = da1 * (1 - np.power(a1, 2))

    dW1 = (1/m) * np.dot(dz1, X_T.T)
    db1 = (1/m) * np.sum(dz1, axis=1, keepdims=True)

    # Update
    W1 -= 0.01 * dW1
    b1 -= 0.01 * db1
    W2 -= 0.01 * dW2
    b2 -= 0.01 * db2

    if i % 10 == 0:
        print(f"Epoch {i}, Loss: {loss}")

Epoch 0, Loss: 2.3018342229200153
Epoch 10, Loss: 2.300095230898441
Epoch 20, Loss: 2.2983676010886462
Epoch 30, Loss: 2.2966452279154628
Epoch 40, Loss: 2.294921886994808
Epoch 50, Loss: 2.2931911829274294
Epoch 60, Loss: 2.2914464961317047
Epoch 70, Loss: 2.289680928360127
Epoch 80, Loss: 2.2878872465660716
Epoch 90, Loss: 2.2860578248136436
